<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L14_Model_Evaluation_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L14: Model Evaluation Metrics - Measuring LLM Performance

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 14 of 15

---

## Learning Objectives

By the end of this lesson, you will:
- Understand key evaluation metrics for language models
- Compute perplexity for language model quality
- Use BLEU for translation and generation evaluation
- Apply ROUGE for summarization and recall-oriented tasks
- Calculate F1, precision, and recall for classification tasks
- Compare metrics and choose the right one for your task
- Implement evaluation pipelines on sample model outputs

---

## Concept: Why Evaluate LLMs?

**Evaluation** tells us how well a model performs on a task. Different tasks need different metrics:

- **Language modeling** → Perplexity (how surprised the model is by text)
- **Translation / generation** → BLEU (n-gram overlap with reference)
- **Summarization** → ROUGE (recall of important content)
- **Classification** → F1, precision, recall (correct labels)

No single metric captures "quality"; we combine metrics and human evaluation.

---

In [ ]:
# Step 1: Install and Import Required Libraries
!pip install transformers torch accelerate evaluate nltk rouge-score scikit-learn -q

import torch
import math
from transformers import AutoTokenizer, AutoModelForCausalLM
from evaluate import load
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data for BLEU (run once)
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

print("Libraries imported successfully!")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: Perplexity

**Perplexity** measures how well a language model predicts a sequence. Lower is better.

- Formula: \( \text{PPL} = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log p(x_i)\right) \)
- Intuition: "How surprised is the model by this text?"
- Use for: comparing LMs, measuring fit to a corpus

---

In [ ]:
# Step 2: Compute Perplexity

print("Perplexity Calculation\n")
print("=" * 60)

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

def compute_perplexity(model, tokenizer, text, device=None):
    """Compute perplexity for a string."""
    if device is None:
        device = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_ids = inputs["input_ids"]
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
    loss = outputs.loss
    return math.exp(loss.item())

# Example texts
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is a subset of artificial intelligence.",
    "xyz abc qwerty random words sequence.",
]

model = model.to("cuda" if torch.cuda.is_available() else "cpu")
for i, text in enumerate(texts, 1):
    ppl = compute_perplexity(model, tokenizer, text)
    print(f"Text {i}: {text[:50]}..." if len(text) > 50 else f"Text {i}: {text}")
    print(f"Perplexity: {ppl:.2f}\n")

print("=" * 60)
print("Lower perplexity = model finds the text more likely (better fit).")

## Part 2: BLEU Score

**BLEU** (Bilingual Evaluation Understudy) measures n-gram overlap between model output and reference(s).

- Range: 0–1 (often reported 0–100). Higher is better.
- Uses 1-gram to 4-gram precision + brevity penalty.
- Use for: translation, any generation with reference text.

---

In [ ]:
# Step 3: BLEU Score

print("BLEU Score\n")
print("=" * 60)

bleu = load("bleu")

# Reference and candidate (e.g. translation)
references = [["The cat is sitting on the mat."]]  # list of reference token lists per prediction
predictions = ["The cat is sitting on the mat."]   # exact match

result = bleu.compute(predictions=predictions, references=references)
print(f"Reference: {references[0][0]}")
print(f"Prediction: {predictions[0]}")
print(f"BLEU: {result['bleu']:.4f}\n")

# Slightly different prediction
predictions2 = ["A cat is sitting on the mat."]
result2 = bleu.compute(predictions=predictions2, references=references)
print(f"Prediction: {predictions2[0]}")
print(f"BLEU: {result2['bleu']:.4f}\n")

# Poor prediction
predictions3 = ["The dog runs in the park."]
result3 = bleu.compute(predictions=predictions3, references=references)
print(f"Prediction: {predictions3[0]}")
print(f"BLEU: {result3['bleu']:.4f}")

print("=" * 60)

## Part 3: ROUGE Score

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures overlap with references (recall/f1).

- ROUGE-1, ROUGE-2: unigram/bigram overlap
- ROUGE-L: longest common subsequence
- Use for: summarization, any recall-oriented generation

---

In [ ]:
# Step 4: ROUGE Score

print("ROUGE Score\n")
print("=" * 60)

rouge = load("rouge")

reference_summary = "Machine learning is a branch of artificial intelligence."
predicted_summary = "Machine learning is part of AI."

result = rouge.compute(predictions=[predicted_summary], references=[reference_summary])
print(f"Reference: {reference_summary}")
print(f"Prediction: {predicted_summary}")
print(f"ROUGE-1: {result['rouge1']}")
print(f"ROUGE-2: {result['rouge2']}")
print(f"ROUGE-L: {result['rougeL']}")
print(f"ROUGE-Lsum: {result['rougeLsum']}")

print("=" * 60)

## Part 4: F1, Precision, and Recall (Classification)

For **classification** (e.g. sentiment, intent):

- **Precision**: Of all predicted positive, how many were correct?
- **Recall**: Of all actual positive, how many did we find?
- **F1**: Harmonic mean of precision and recall.

---

In [ ]:
# Step 5: F1, Precision, Recall

from sklearn.metrics import precision_recall_fscore_support, classification_report, confusion_matrix

print("Classification Metrics\n")
print("=" * 60)

# Example: sentiment predictions vs ground truth
y_true = ["positive", "negative", "positive", "neutral", "negative", "positive"]
y_pred = ["positive", "negative", "negative", "neutral", "negative", "positive"]

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
print(f"Weighted Precision: {precision:.3f}")
print(f"Weighted Recall:    {recall:.3f}")
print(f"Weighted F1:        {f1:.3f}\n")

print("Per-class metrics:")
print(classification_report(y_true, y_pred))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred))
print("=" * 60)

## Part 5: End-to-End Evaluation Example

Combine metrics: generate with a model, then evaluate with BLEU/ROUGE and (if applicable) classification metrics.

---

In [ ]:
# Step 6: Evaluate Model Generation

print("Evaluating Generated Text\n")
print("=" * 60)

from transformers import AutoModelForCausalLM

prompt = "The capital of France is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"Generated: {generated}\n")

# Compare to reference with BLEU
reference = "The capital of France is Paris."
ref_list = [[reference]]
pred_list = [generated]
bleu_result = bleu.compute(predictions=pred_list, references=ref_list)
rouge_result = rouge.compute(predictions=pred_list, references=[reference])
print(f"Reference: {reference}")
print(f"BLEU: {bleu_result['bleu']:.4f}")
print(f"ROUGE-1: {rouge_result['rouge1']}")
print("=" * 60)

## Exercises

### Exercise 1: Perplexity Comparison
Compute perplexity for 5 different sentences (e.g. grammatical vs ungrammatical, in-domain vs out-of-domain). Compare and explain the results.

### Exercise 2: BLEU on Multiple References
Use 2–3 references per sentence and compute BLEU. Compare with single-reference BLEU and discuss when multiple references help.

### Exercise 3: Custom Evaluation Metric
Implement a simple custom metric (e.g. word overlap rate, average word length match) and compare it with BLEU/ROUGE on a few examples.

### Exercise 4: Classification Pipeline
Use a small set of labeled sentences, get model predictions (e.g. zero-shot or few-shot), and compute precision, recall, and F1. Try different prompts and see how metrics change.

### Exercise 5: Evaluation Report
For a fixed prompt and model, generate 10 completions. For each, compute BLEU and ROUGE against one reference. Summarize mean and variance of the metrics.

---

## Key Takeaways

1. **Perplexity** measures how well a LM predicts text; lower is better.
2. **BLEU** measures n-gram overlap with references; good for translation/generation.
3. **ROUGE** measures overlap (recall-oriented); good for summarization.
4. **F1, precision, recall** are standard for classification tasks.
5. Choose metrics that match your task and always pair with human evaluation when possible.

---

## Additional Resources

- **Papers:** *BLEU* (Papineni et al.), *ROUGE* (Lin, 2004), *Perplexity* in LM literature
- **Docs:** [HuggingFace Evaluate](https://huggingface.co/docs/evaluate), [scikit-learn metrics](https://scikit-learn.org/stable/modules/model_evaluation.html)

---

## Next Lesson

**L15: Project - Simple Chatbot** – Build a chatbot with conversation management, context handling, and response generation using concepts from L1–L14.

---

**Congratulations! You can now evaluate LLM outputs with standard metrics.**